# NLP Classification of MIMIC-CXR Radiology Reports

Sentiment-based urgency classification of chest X-ray reports to support real-time triage. This notebook implements the NLP pipeline that feeds the Power BI urgency-triage dashboard and the Tableau efficiency dashboard.

**Inputs:** MIMIC-CXR free-text radiology reports (access via PhysioNet).  
**Outputs:** Per-report urgency score, urgency flag, and aggregate efficiency metrics.

In [ ]:
import os
import yaml
import pandas as pd
import numpy as np
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

INPUT_PATH = config['data']['mimic_cxr_path']
OUTPUT_PATH = config['data']['output_path']
URGENCY_THRESHOLD = config['nlp']['urgency_threshold']
MAX_LEN = config['nlp']['max_report_length']

In [ ]:
if not os.path.exists(INPUT_PATH):
    print(f'MIMIC-CXR file not found at {INPUT_PATH}; falling back to sample_reports.csv')
    df = pd.read_csv('../data/sample_reports.csv')
else:
    df = pd.read_csv(INPUT_PATH)

df['report_text'] = df['report_text'].fillna('').str.slice(0, MAX_LEN)
print(df.shape)
df.head()

In [ ]:
analyzer = SentimentIntensityAnalyzer()

URGENT_TERMS = [
    'pneumothorax', 'tension', 'massive', 'acute', 'hemorrhage',
    'severe', 'rapidly', 'worsening', 'critical', 'emergent'
]

def urgency_score(text: str) -> float:
    text_lower = text.lower()
    sentiment = analyzer.polarity_scores(text_lower)
    negative_pressure = sentiment['neg']
    keyword_hits = sum(1 for term in URGENT_TERMS if term in text_lower)
    keyword_signal = min(keyword_hits / 3.0, 1.0)
    return float(0.6 * negative_pressure + 0.4 * keyword_signal)

df['urgency_score'] = df['report_text'].apply(urgency_score)
df['urgent_flag'] = (df['urgency_score'] >= URGENCY_THRESHOLD).astype(int)
df.head()

In [ ]:
summary = df.groupby('case_type').agg(
    n=('urgency_score', 'size'),
    mean_urgency=('urgency_score', 'mean'),
    pct_urgent=('urgent_flag', 'mean'),
    mean_turnaround_min=('turnaround_minutes', 'mean')
).round(3)
summary

In [ ]:
ai_assisted = df[df['ai_assisted'] == 1]['turnaround_minutes'].mean()
unassisted = df[df['ai_assisted'] == 0]['turnaround_minutes'].mean()
if unassisted:
    improvement = (unassisted - ai_assisted) / unassisted * 100
    print(f'Mean turnaround AI-assisted: {ai_assisted:.1f} min')
    print(f'Mean turnaround unassisted: {unassisted:.1f} min')
    print(f'Efficiency improvement: {improvement:.1f}%')

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f'Wrote {len(df)} rows to {OUTPUT_PATH}')